In [1]:
"""
Created on Fri Feb 15 11:43 2021

This script computes the ice shelf mask, and box and plume characteristics from the NEMO data for futher procedure.
It uses functions from the package multimelt.

@author: Clara Burgard
"""

'\nCreated on Fri Feb 15 11:43 2021\n\nThis script computes the ice shelf mask, and box and plume characteristics from the NEMO data for futher procedure.\nIt uses functions from the package multimelt.\n\n@author: Clara Burgard\n'

In [4]:
import xarray as xr
import numpy as np
from pyproj import Transformer
import pandas as pd
from tqdm.notebook import trange, tqdm
import multimelt.plume_functions as pf
import multimelt.box_functions as bf
import multimelt.useful_functions as uf
import multimelt.create_isf_mask_functions as isfmf

In [5]:
%matplotlib inline

In [6]:
map_lim = [-3000000,3000000]

# if using dask, put this to True. But no guarantee that it works, I tried it out but was not very satisfied with it
chunk_size = False

In [7]:
nemo_run = 'OPM018'

READ IN DATA

In [16]:
inputpath_data='/bettik/burgardc/SCRIPTS/basal_melt_param/data/interim/NEMO_eORCA025.L121_'+nemo_run+'_ANT_STEREO/'
inputpath_metadata='/bettik/burgardc/SCRIPTS/basal_melt_param/data/raw/MASK_METADATA/'
#outputpath_mask='/bettik/burgardc/SCRIPTS/basal_melt_param/data/interim/ANTARCTICA_IS_MASKS/nemo_5km_'+nemo_run+'/'
outputpath_mask ='/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/data/interim/ANTARCTICA_IS_MASKS/nemo_5km_'+nemo_run+'/'
#outputpath_mask_orig='/bettik/burgardc/SCRIPTS/basal_melt_param/data/interim/ANTARCTICA_IS_MASKS/nemo_5km/'
#outputpath_boxes = '/bettik/burgardc/SCRIPTS/basal_melt_param/data/interim/BOXES/nemo_5km_'+nemo_run+'/'
outputpath_boxes = '/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/data/interim/ANTARCTICA_IS_MASKS/nemo_5km_'+nemo_run+'/'
#outputpath_plumes = '/bettik/burgardc/SCRIPTS/basal_melt_param/data/interim/PLUMES/nemo_5km_'+nemo_run+'/'
outputpath_plumes = '/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/data/interim/ANTARCTICA_IS_MASKS/nemo_5km_'+nemo_run+'/'

file_mask_orig = xr.open_dataset(inputpath_data+'other_mask_vars_Ant_stereo.nc')
file_mask_orig_cut = uf.cut_domain_stereo(file_mask_orig, map_lim, map_lim)
file_mask = xr.open_dataset(inputpath_data+'custom_lsmask_Ant_stereo_clean.nc')#, chunks={'x': chunk_size, 'y': chunk_size})
file_mask_cut = uf.cut_domain_stereo(file_mask, map_lim, map_lim)
file_other = xr.open_dataset(inputpath_data+'corrected_draft_bathy_isf.nc')#, chunks={'x': chunk_size, 'y': chunk_size})
file_other_cut = uf.cut_domain_stereo(file_other, map_lim, map_lim)
file_conc = xr.open_dataset(inputpath_data+'isfdraft_conc_Ant_stereo.nc')
file_conc_cut = uf.cut_domain_stereo(file_conc, map_lim, map_lim)

In [9]:
file_bed_orig = file_mask_orig_cut['bathy_metry'] 
file_draft = file_other_cut['corrected_isfdraft'] 
file_msk = file_mask_cut['ls_mask012'] #0 = ocean, 1 = ice shelves, 2 = grounded ice
file_isf_conc = file_conc_cut['isfdraft_conc']

xx = file_mask_cut['x']
yy = file_mask_cut['y']


Create the masks for ice shelves/ground/pinning points/grounding line

In [10]:
whole_ds = isfmf.create_mask_and_metadata_isf(file_msk, -1*file_bed_orig, file_msk, -1*file_draft, file_isf_conc, False, 
                                              inputpath_metadata+'lonlat_masks.txt', outputpath_mask, 
                                              inputpath_metadata + 'iceshelves_metadata_Nico.txt', 
                                              inputpath_metadata+'GL_flux_rignot13.csv',
                                              ground_point ='no',dist=40, add_fac=120)

# Write to netcdf
print('------- WRITE TO NETCDF -----------')
#whole_ds.to_netcdf(outputpath_mask + 'nemo_5km_isf_masks_and_info_and_distance_new.nc','w') # separated Filchner and Ronne (before review)
whole_ds.to_netcdf(outputpath_mask + 'nemo_5km_isf_masks_and_info_and_distance_new_oneFRIS.nc','w')

--------- PREPARE THE MASKS --------------
handling coordinates
Reading in latlon boundaries
Define the regional masks
Define ground_mask
Distance to South Pole for initial ground square = 40
Additional number of iterations = 120


  0%|          | 0/729 [00:00<?, ?it/s]

Define grounding line
Grounding line is the first point on the ground: no
Define front
Define pinning points


  0%|          | 0/25 [00:00<?, ?it/s]

Define pinning point boundaries
Merge into one netcdf
--------- PREPARE THE METADATA --------------
Prepare csv with metadata


/bettik/ockendeh/SCRIPTS/multimelt/multimelt/create_isf_mask_functions.py:911: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['isf_name'].loc[11] = 'Filchner-Ronne'
/bettik/ockendeh/SCRIPTS/multimelt/multimelt/create_isf_mask_functions.py:912: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['isf_melt'].loc[11] = df['isf_melt'].loc[11] + df['isf_melt'].loc[21]
/bettik/ockendeh/SCRIPTS/multimelt/multimelt/create_isf_mask_functions.py:913: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_gu

--------- COMBINE MASK AND METADATA --------------
--------- COMPUTE DISTANCE TO GROUNDING LINE AND ICE FRONT --------------


/bettik/ockendeh/SCRIPTS/multimelt/multimelt/create_isf_mask_functions.py:1032: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['front_max_lon'].loc[10] = lon.where(mask_front == 10).where(lon < -100).max()
/bettik/ockendeh/SCRIPTS/multimelt/multimelt/create_isf_mask_functions.py:1033: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['front_min_lon'].loc[10] = lon.where(mask_front == 10).where(lon > 100).min()


  0%|          | 0/130 [00:00<?, ?it/s]

------- WRITE TO NETCDF -----------


Prepare the box characteristics (writes the output directly to files)

In [11]:
#whole_ds = xr.open_dataset(outputpath_mask + 'nemo_5km_isf_masks_and_info_and_distance_new.nc') # separated Filchner and Ronne (before review)
whole_ds = xr.open_dataset(outputpath_mask + 'nemo_5km_isf_masks_and_info_and_distance_new_oneFRIS.nc')

nonnan_Nisf = whole_ds['Nisf'].where(np.isfinite(whole_ds['front_bot_depth_max']), drop=True).astype(int)
file_isf_nonnan = whole_ds.sel(Nisf=nonnan_Nisf)
large_isf = file_isf_nonnan['Nisf'].where(file_isf_nonnan['isf_area_here'] >= 2500, drop=True)
file_isf = file_isf_nonnan.sel(Nisf=large_isf)

In [14]:
isf_var_of_int = whole_ds[['ISF_mask', 'GL_mask', 'dGL', 'dIF', 'latitude', 'longitude', 'isf_name']]
out_2D, out_1D = bf.box_charac_file(file_isf['Nisf'],isf_var_of_int, -1*file_draft, file_isf_conc, outputpath_boxes, max_nb_box=10)
#out_2D.to_netcdf(outputpath_boxes + 'nemo_5km_boxes_2D.nc') # separated Filchner and Ronne (before review)
#out_1D.to_netcdf(outputpath_boxes + 'nemo_5km_boxes_1D.nc') # separated Filchner and Ronne (before review)
out_2D.to_netcdf(outputpath_boxes + 'nemo_5km_boxes_2D_oneFRIS.nc')
out_1D.to_netcdf(outputpath_boxes + 'nemo_5km_boxes_1D_oneFRIS.nc')

  0%|          | 0/35 [00:00<?, ?it/s]

/home/ockendeh/miniconda3/envs/nnets_py38/lib/python3.8/site-packages/dask/core.py:119: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/ockendeh/miniconda3/envs/nnets_py38/lib/python3.8/site-packages/dask/core.py:119: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/ockendeh/miniconda3/envs/nnets_py38/lib/python3.8/site-packages/dask/core.py:119: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/ockendeh/miniconda3/envs/nnets_py38/lib/python3.8/site-packages/dask/core.py:119: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/ockendeh/miniconda3/envs/nnets_py38/lib/python3.8/site-packages/dask/core.py:119: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/ockendeh/minic

PermissionError: [Errno 13] Permission denied: b'/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/data/interim/nemo_5km_OPM018/nemo_5km_boxes_2D_oneFRIS.nc'

Prepare the plume characteristics

In [17]:
plume_param_options = ['simple','lazero', 'appenB']

plume_var_of_int = file_isf[['ISF_mask', 'GL_mask', 'IF_mask', 'dIF', 'dGL_dIF', 'latitude', 'longitude', 'front_ice_depth_avg']]

# Compute the ice draft
ice_draft_pos = file_draft
ice_draft_neg = -1*ice_draft_pos


plume_charac = pf.prepare_plume_charac(plume_param_options, ice_draft_pos, plume_var_of_int)
print('------ WRITE TO NETCDF -------')
#plume_charac.to_netcdf(outputpath_plumes+'nemo_5km_plume_characteristics.nc')
plume_charac.to_netcdf(outputpath_plumes+'nemo_5km_plume_characteristics_oneFRIS.nc')

----------------- PREPARATION GL MAX --------------


  0%|          | 0/35 [00:00<?, ?it/s]

  0%|          | 0/35 [00:00<?, ?it/s]

UnboundLocalError: local variable 'alpha' referenced before assignment

CHECK RESULTS

In [ ]:
whole_ds = xr.open_dataset(outputpath_mask + 'nemo_5km_isf_masks_and_info_and_distance_new_oneFRIS.nc')


In [ ]:
nonnan_Nisf = whole_ds['Nisf'].where(np.isfinite(whole_ds['front_bot_depth_max']), drop=True).astype(int)
file_isf_nonnan = whole_ds.sel(Nisf=nonnan_Nisf)
large_isf = file_isf_nonnan['Nisf'].where(file_isf_nonnan['isf_area_here'] >= 2500, drop=True)
file_isf = file_isf_nonnan.sel(Nisf=large_isf)

In [ ]:
isf_mask = file_isf['ISF_mask'].where(file_isf['ISF_mask'] == file_isf.Nisf).sum('Nisf')
isf_mask = isf_mask.where(isf_mask)

In [ ]:
GL_flux = file_isf['GL_flux']

In [ ]:
GL_flux_new = GL_flux.where(isf_mask==file_isf.Nisf).sum('Nisf')
GL_flux_new = GL_flux_new.where(GL_flux_new)